# Move Abs Notebook

Absolute one-axis moves through `kohdalab.api.Experiment`.

Edit `config_path`, `coordinate`, and `targets` before running on hardware. In notebooks `Experiment` uses `auto_connect=True` by default, so a missing required device can be connected automatically when the move runs.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src" / "kohdalab").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/kohdalab.")


repo_root = find_repo_root()
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
config_path = repo_root / "src" / "kohdalab" / "config" / "trkr_config_kikuchi.json"

from kohdalab.api import Experiment, format_move_abs_row, load_config, move_abs_row_from_position

config = load_config(config_path)
experiment = Experiment(config)
zero = config["measurements"]["move_abs"]["zero"]
configured_targets = config["measurements"]["move_abs"].get("targets", {})


In [ ]:
coordinate = "measurement"
targets = {
    "t": float(configured_targets.get("t", zero.get("t_ps", 0.0))),
    "x": float(configured_targets.get("x", zero.get("x_um", 0.0))),
    "y": float(configured_targets.get("y", zero.get("y_um", 0.0))),
}
axes_to_move = ["t", "x", "y"]


In [ ]:
rows = []
for axis in axes_to_move:
    target = targets[axis]
    if axis == "t":
        position = experiment.move_delay_stage(target, coordinate=coordinate)
    else:
        position = experiment.move_scanner(axis, target, coordinate=coordinate)
    row = move_abs_row_from_position(axis, position, target=target, coordinate=coordinate, zero=zero)
    rows.append(row)
    print(format_move_abs_row(row, index=len(rows), total=len(axes_to_move)), flush=True)

display(pd.DataFrame(rows))


In [ ]:
# Run this when you are done with the notebook session.
# experiment.disconnect_all()
